# Explore EPANET through MCP and LangChain

This notebook is a small, complete introduction to the EPANET MCP server. It uses the LangChain MCP adapter to discover and invoke tools directly. No LLM and no API key are required.

The walkthrough performs these operations in order:

1. Connect to the EPANET MCP server.
2. List every available MCP tool and its arguments.
3. List every EPANET network bundled with ePyT.
4. Select and load `Net1.inp`.
5. Set a new pipe diameter.
6. Unload the network and close the server session safely.

## 1. Prerequisites

From the repository root, install the project and the LangChain MCP adapter into the environment used by this notebook:

```bash
uv sync --extra examples
```

Restart the kernel after installation if necessary. An OpenAI key is not needed because this example calls the MCP tools directly.

## 2. Understand the session lifecycle

A loaded EPANET network lives in the MCP server process, so all stateful calls must use the same server session:

```text
open MCP session
    -> discover tools
    -> list bundled networks
    -> load network
    -> inspect network
    -> unload network
close MCP session
```

The complete workflow runs inside one `async with client.session(...)` block. This is intentional: `client.get_tools()` may create a new session for each call, and splitting an AnyIO session context across notebook tasks can cause cancellation-scope errors.

## 3. Define the MCP client and reusable helpers

The server is launched with the same Python interpreter as the notebook. The helper below converts MCP text content into ordinary Python dictionaries and lists.

In [13]:
import json
import sys

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools


def decode_mcp_result(result):
    """Convert LangChain MCP content blocks into normal Python data."""
    if isinstance(result, list):
        text_parts = []
        for block in result:
            if isinstance(block, dict) and block.get("type") == "text":
                text_parts.append(block.get("text", ""))
            elif getattr(block, "type", None) == "text":
                text_parts.append(getattr(block, "text", ""))

        if len(text_parts) == 1:
            try:
                return json.loads(text_parts[0])
            except json.JSONDecodeError:
                return text_parts[0]
        return text_parts

    if isinstance(result, str):
        try:
            return json.loads(result)
        except json.JSONDecodeError:
            return result

    return result


def print_section(title):
    """Print a visible section heading in the notebook output."""
    line = "=" * len(title)
    print(f"\n{title}\n{line}")


def short_description(tool):
    """Return the first documentation paragraph as one readable line."""
    paragraph = []
    for line in (tool.description or "").splitlines():
        line = line.strip()
        if not line and paragraph:
            break
        if line:
            paragraph.append(line)
    return " ".join(paragraph) or "No description provided."


def format_tool_arguments(tool):
    """Mark optional arguments so the generated tool list is unambiguous."""
    schema = tool.args_schema if isinstance(tool.args_schema, dict) else {}
    required = set(schema.get("required", []))
    arguments = [
        name if name in required else f"{name}=optional"
        for name in tool.args
    ]
    return ", ".join(arguments) or "no arguments"


def describe_network(summary):
    """Turn the machine-readable network summary into plain language."""
    nodes = summary["nodes"]
    links = summary["links"]
    duration_hours = summary["simulation_duration_s"] / 3600
    timestep_minutes = summary["hydraulic_timestep_s"] / 60

    return (
        f"Net1 contains {nodes['total']} nodes: {nodes['junctions']} junctions, "
        f"{nodes['tanks']} tank, and {nodes['reservoirs']} reservoir. It has "
        f"{links['total']} links: {links['pipes']} pipes, {links['pumps']} pump, "
        f"and {links['valves']} valves. Flow is expressed in "
        f"{summary['flow_units']}. The configured simulation lasts "
        f"{duration_hours:g} hours with a {timestep_minutes:g}-minute hydraulic "
        f"timestep. The model also defines {summary['patterns']} pattern, "
        f"{summary['curves']} curve, and {summary['simple_controls']} simple controls."
    )


client = MultiServerMCPClient(
    {
        "epanet": {
            "transport": "stdio",
            "command": sys.executable,
            "args": ["-m", "epanet_mcp.server"],
        }
    }
)

## 4. Show available tools

The session will show all tool descriptions. The `finally` block unloads the network even if summary generation fails. The function returns the collected Python data so later cells can inspect it without keeping the server open.

In [14]:
async def show_available_tools():
    async with client.session("epanet") as session:
        tools = await load_mcp_tools(session)
        print("Connected to the EPANET MCP server.")

        print_section(f"1. Available MCP tools ({len(tools)})")
        for number, tool in enumerate(tools, start=1):
            arguments = format_tool_arguments(tool)
            print(f"{number:2}. {tool.name}({arguments})")
            print(f"    {short_description(tool)}")

        tools_gathered = [tool.name for tool in tools]

    print("MCP session closed.")
    return tools_gathered

demo = await show_available_tools()

Connected to the EPANET MCP server.

1. Available MCP tools (45)
 1. load_network(path, network_id=optional)
    Load an EPANET .inp file into a named session.
 2. unload_network(network_id)
    Unload a previously loaded network and free its resources.
 3. list_networks(no arguments)
    Return the ids of all currently loaded networks.
 4. list_bundled_networks(no arguments)
    List every .inp example file bundled with ePyT (usable with load_network).
 5. get_network_summary(network_id)
    Return a high-level summary of a loaded network.
 6. get_nodes(network_id)
    Return all nodes (junctions, tanks, reservoirs) with their attributes.
 7. get_links(network_id)
    Return all links (pipes, pumps, valves) with their attributes.
 8. get_patterns(network_id)
    Return all demand / operational patterns defined in the network.
 9. get_controls(network_id)
    Return all simple controls defined in the network.
10. get_curves(network_id)
    Return all curves (pump head, efficiency, volu

## 5. Run the walkthrough

This produces the requested order: all tools first, bundled networks second, and the Net1 summary last.

In [15]:
from pathlib import Path


async def set_pipe_diameter(
    network_path: str = "Net1.inp",
    pipe_id: str = "10",
    diameter: float = 150.0,
    output_path: str = "./Net1_modified.inp",
):
    async with client.session("epanet") as session:
        tools = {
            tool.name: tool
            for tool in await load_mcp_tools(session)
        }

        # 1. Load network
        loaded = await tools["load_network"].ainvoke({
            "path": network_path,
            "network_id": "my_network",
        })
        print("Loaded:", loaded)

        # 2. Modify pipe diameter
        modified = await tools["set_pipe_diameter"].ainvoke({
            "network_id": "my_network",
            "pipe_id": pipe_id,
            "diameter": diameter,
        })
        print("Modified:", modified)

        # 3. Save modified network
        saved = await tools["save_network"].ainvoke({
            "network_id": "my_network",
            "output_path": str(Path(output_path).resolve()),
        })
        print("Saved:", saved)

        finished = await tools["unload_network"].ainvoke({
            "network_id": "my_network",
        })

        return saved

result = await set_pipe_diameter(
    network_path="Net1.inp",
    pipe_id="10",
    diameter=150.0,
    output_path="./Net1_modified.inp",
)

Loaded: [{'type': 'text', 'text': '{\n  "network_id": "my_network",\n  "path": "/Users/oriol.alas/PROJECTS/EPANET/epanet-mcp-server/.venv/lib/python3.11/site-packages/epyt/networks/asce-tf-wdst/Net1.inp",\n  "summary": {\n    "nodes": {\n      "total": 11,\n      "junctions": 9,\n      "tanks": 1,\n      "reservoirs": 1\n    },\n    "links": {\n      "total": 13,\n      "pipes": 12,\n      "pumps": 1,\n      "valves": 0\n    },\n    "patterns": 1,\n    "curves": 1,\n    "simple_controls": 2,\n    "rule_based_controls": 0,\n    "flow_units": "GPM",\n    "simulation_duration_s": 86400,\n    "hydraulic_timestep_s": 3600\n  }\n}', 'id': 'lc_27353df7-e00b-4eec-8fea-dd13aa3a4976'}]
Modified: [{'type': 'text', 'text': '{\n  "network_id": "my_network",\n  "pipe_id": "10",\n  "diameter": 150.0\n}', 'id': 'lc_51131f11-0fdf-4a32-8257-272d3bfb3f7d'}]
Saved: [{'type': 'text', 'text': '{\n  "saved": true,\n  "output_path": "/Users/oriol.alas/PROJECTS/EPANET/epanet-mcp-server/examples/Net1_modified.i

## 6. Check the pipe diameter has changed

The MCP session is now closed, but the results returned by the tools remain available. Moreover, the result itself is saved in Net1_modified. This makes them easy to analyze, visualize, or pass to an LLM later.

In [16]:
from epyt import epanet

d = epanet("./Net1_modified.inp")
print(d.getLinkNameID())
print(d.getLinkDiameter())
d = epanet("./Net1.inp")
print(d.getLinkNameID())
print(d.getLinkDiameter())

EPANET version 20305 loaded (EPyT version v2.3.5.0 - Last Update: 20/02/2026).
Publication DOI: https://zenodo.org/records/18484878
Input File Net1_modified.inp loaded successfully.

['10', '11', '12', '21', '22', '31', '110', '111', '112', '113', '121', '122', '9']
[150.  14.  10.  10.  12.   6.  18.  10.  12.   8.   8.   6.   0.]
EPANET version 20305 loaded (EPyT version v2.3.5.0 - Last Update: 20/02/2026).
Publication DOI: https://zenodo.org/records/18484878
Input File Net1.inp loaded successfully.

['10', '11', '12', '21', '22', '31', '110', '111', '112', '113', '121', '122', '9']
[18. 14. 10. 10. 12.  6. 18. 10. 12.  8.  8.  6.  0.]


## Next steps

Use the same pattern to call other tools inside a persistent session. For example:

- `get_nodes` and `get_links` inspect individual assets.
- `run_hydraulic_simulation` calculates pressure and flow over time.
- modification tools change the loaded in-memory network.
- scenario tools clone the baseline before applying what-if changes.
- `save_network` persists in-memory changes before the network is unloaded.

When adding an LLM agent, pass the `tools` returned by `load_mcp_tools(session)` to the agent while that same MCP session remains open. Store API keys in environment variables—never directly in a notebook.